# 03 — Comparação entre LLM pura, RAG e agente

**PassoLivre AI | Generative AI for Engineering — FIAP**

Este notebook apresenta **saídas reais previamente registradas** no experimento independente de 16 perguntas × 3 modos. O arquivo de resultados é lido do repositório e confrontado com `placar.json`. Nenhuma resposta da LLM é simulada e a visualização não depende do Ollama estar aberto.

- Puro: Qwen sem documentos do projeto.
- RAG: Qwen com até três trechos recuperados.
- Agente: RAG com regras e consulta exata de ocorrências.

O agente não altera os pesos do Qwen (não houve LoRA/fine-tuning). As métricas de correção e alucinação são da auditoria humana registrada no CSV.

In [8]:
from pathlib import Path
import json
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

In [9]:
import csv
from collections import Counter

def carregar_csv(caminho):
    with caminho.open(encoding="utf-8-sig", newline="") as f:
        cabecalho = f.readline()
        f.seek(0)
        delimitador = ";" if cabecalho.count(";") > cabecalho.count(",") else ","
        return list(csv.DictReader(f, delimiter=delimitador))

def e_true(valor):
    return str(valor).strip().casefold() in ("true", "verdadeiro", "1")

rodadas = ROOT / "avaliacao/rodadas"
preferida = rodadas / "avaliacao_independente_e68b6315fca6"
opcoes = [preferida] if (preferida / "placar.json").is_file() else sorted(rodadas.glob("avaliacao_independente_*"))
if len(opcoes) != 1:
    raise FileNotFoundError("Não foi possível identificar uma única rodada independente com placar.json.")
rodada = opcoes[0]
csv_path = rodada / "resultados_revisados_v2.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Arquivo final de auditoria não encontrado: {csv_path.name}")
resultados = carregar_csv(csv_path)
placar = json.loads((rodada / "placar.json").read_text(encoding="utf-8"))
modos = ("puro", "rag", "agente")
pares = [(r["id"], r["modo"]) for r in resultados]
assert len(resultados) == 48 and len(set(pares)) == 48, "Esperadas 48 respostas únicas (16 por modo)."
assert all(r["correcao_humana"] in ("0", "1") and r["alucinacao_humana"] in ("0", "1") for r in resultados), "Há classificações humanas em aberto."
print(f"Rodada: {rodada.name}")
print(f"Respostas auditadas: {len(resultados)} | Perguntas: {len(set(r['id'] for r in resultados))}")

Rodada: avaliacao_independente_e68b6315fca6
Respostas auditadas: 48 | Perguntas: 16


## Placar auditado

Acurácia é a proporção de respostas classificadas como corretas. Taxa de alucinação é a proporção de respostas com pelo menos uma afirmação falsa ou sem suporte, segundo a rubrica humana. A taxa de recusas conforme o gabarito mede o campo booleano, e não a qualidade total do texto.

In [10]:
calculadas = {}
print(f"{'Modo':9} {'Corretas':>10} {'Alucinações':>12} {'Acurácia':>10} {'Taxa aluc.':>12} {'Recusas conf.':>15}")
for modo in modos:
    r = [x for x in resultados if x["modo"] == modo]
    assert len(r) == 16
    acertos = sum(int(x["correcao_humana"]) for x in r)
    alucinacoes = sum(int(x["alucinacao_humana"]) for x in r)
    recusas = sum(e_true(x["recusa_correta"]) for x in r)
    acuracia, taxa = acertos / 16, alucinacoes / 16
    registrado = placar["modos"][modo]
    assert abs(acuracia - registrado["acuracia_humana"]) < 1e-4, f"Placar não coincide com as revisões: {modo}, acurácia"
    assert abs(taxa - registrado["taxa_alucinacao_humana"]) < 1e-4, f"Placar não coincide com as revisões: {modo}, alucinação"
    assert abs(recusas / 16 - registrado["recusas_conforme_gabarito"]) < 1e-4, f"Placar não coincide: {modo}, recusas"
    calculadas[modo] = (acuracia, taxa, recusas / 16)
    print(f"{modo:9} {acertos:>8}/16 {alucinacoes:>10}/16 {acuracia:>9.2%} {taxa:>11.2%} {recusas / 16:>14.2%}")
print("Conferência CSV ↔ placar.json: OK")

Modo        Corretas  Alucinações   Acurácia   Taxa aluc.   Recusas conf.
puro             3/16          4/16    18.75%      25.00%         68.75%
rag             11/16          1/16    68.75%       6.25%         75.00%
agente          12/16          2/16    75.00%      12.50%         93.75%
Conferência CSV ↔ placar.json: OK


In [11]:
try:
    import matplotlib.pyplot as plt
    labels = ("Puro", "RAG", "Agente")
    valores = [calculadas[m] for m in modos]
    pos = list(range(3))
    largura = 0.35
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([x - largura/2 for x in pos], [v[0]*100 for v in valores], largura, label="Acurácia humana")
    ax.bar([x + largura/2 for x in pos], [v[1]*100 for v in valores], largura, label="Taxa de alucinação")
    ax.set_xticks(pos, labels)
    ax.set_ylabel("Respostas (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    ax.set_title("Comparação nas 16 perguntas independentes")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Gráfico opcional indisponível; a tabela acima já contém os resultados.")

Gráfico opcional indisponível; a tabela acima já contém os resultados.


## Respostas efetivamente produzidas

Os exemplos seguintes são obtidos do CSV da avaliação, não de novas chamadas ao modelo. Eles permitem comparar uma ocorrência simulada, a interpretação de uma norma e uma solicitação de dado pessoal indisponível.

In [12]:
por_chave = {(r["id"], r["modo"]): r for r in resultados}
for qid in ("N02", "N11", "N15"):
    print("\n" + "="*90)
    print(qid, por_chave[(qid, "puro")]["pergunta"])
    for modo in modos:
        item = por_chave[(qid, modo)]
        print(f"\n[{modo}] Correção humana: {item['correcao_humana']} | Alucinação: {item['alucinacao_humana']} | Fontes: {item['fontes'] or 'nenhuma'}")
        print(item["resposta"][:550])


N02 Um obstáculo pode estar validado e ainda sem resolução? Use o caso da Avenida Modelo.

[puro] Correção humana: 0 | Alucinação: 1 | Fontes: nenhuma
Sim, um obstáculo pode estar validado e ainda sem resolução. Por exemplo, a Avenida Modelo em São Paulo, que foi validada como uma via pública, ainda enfrenta problemas de manutenção e segurança, o que impede sua resolução completa.

[rag] Correção humana: 1 | Alucinação: 0 | Fontes: S002
Sim, um obstáculo pode estar validado e ainda sem resolução, como no caso da Avenida Modelo.

[agente] Correção humana: 1 | Alucinação: 0 | Fontes: S002
Sim, um obstáculo pode estar validado e ainda sem resolução. No caso da Avenida Modelo, o registro simulado de um poste obstruindo a passagem foi validado por um responsável, mas a resolução está em andamento e ainda não foi concluída.

N11 Uma passagem estreita de 80 cm atende automaticamente ao padrão geral da cidade de São Paulo?

[puro] Correção humana: 0 | Alucinação: 1 | Fontes: nenhuma
Não, uma 

## Evidências de cinco retestes

Os cinco erros foram identificados na avaliação original e retestados após alterações nos prompts. A tabela abaixo mostra o campo `recusou` nos modos-alvo, além de um excerto das respostas. Os resultados completos e as causas estão em `docs/erros_e_correcoes.md`.

In [13]:
original = ROOT / "avaliacao/resultados_v1.csv"
retestes = sorted(rodadas.glob("reteste_cinco_erros_*/resultados.csv"))
if original.is_file() and retestes:
    antes = {(r["id"], r["modo"]): r for r in carregar_csv(original)}
    depois = {(r["id"], r["modo"]): r for r in carregar_csv(retestes[0])}
    alvos = (("Q04", "agente"), ("Q07", "agente"), ("Q13", "agente"), ("Q20", "rag"), ("Q24", "agente"))
    for chave in alvos:
        a, d = antes.get(chave), depois.get(chave)
        if a and d:
            print(f"{chave[0]}/{chave[1]} | Antes recusou={a['recusou']}; depois recusou={d['recusou']}")
            print("  Antes:", a["resposta"][:165])
            print("  Depois:", d["resposta"][:165])
        else:
            print("Registro não encontrado:", chave)
else:
    print("Retestes indisponíveis neste ambiente. Consulte docs/erros_e_correcoes.md.")

Q04/agente | Antes recusou=True; depois recusou=False
  Antes: O relato OC003 foi rejeitado porque a verificação não confirmou a ocorrência de desnível. A razão para a rejeição foi que a foto enviada era insuficiente para valida
  Depois: O relato OC003 foi rejeitado porque a foto enviada era insuficiente para confirmar o desnível no local. A verificação rejeitou o relato, conforme estabelecido na pol
Q07/agente | Antes recusou=True; depois recusou=False
  Antes: Não consta informação suficiente para afirmar que visão computacional sozinha confirma um obstáculo.
  Depois: Visão computacional sozinha não confirma um obstáculo. Ela apenas sugere possíveis problemas, que precisam ser revisados por humanos.
Q13/agente | Antes recusou=True; depois recusou=False
  Antes: Não consta informação suficiente para afirmar que o piso irregular no Caminho Demonstrativo 42 continua aberto.
  Depois: O piso irregular no Caminho Demonstrativo 42 foi resolvido, conforme o registro simulado OC005. No ent

## Execução opcional ao vivo (fora do placar)

Para conferir a integração com o Ollama, ajuste `RUN_LLM` para `True` no seu computador e execute a última célula. Isso realiza apenas três novas consultas, **não altera os resultados auditados** nem recalcula o placar. Sem Ollama, deixe `False`.

In [14]:
RUN_LLM = False  # Mantenha False para reabrir no GitHub sem chamar o modelo.
if RUN_LLM:
    from app.core import MODEL, response
    pergunta_ao_vivo = "A ocorrência OC002 já foi resolvida?"
    print("Modelo:", MODEL)
    for modo in modos:
        saida = response(pergunta_ao_vivo, modo)
        print(modo, "→", saida["resposta"], "| fontes:", saida["fontes"])
else:
    print("Nenhuma nova chamada à LLM. Os resultados acima vêm dos arquivos reais da avaliação.")

Nenhuma nova chamada à LLM. Os resultados acima vêm dos arquivos reais da avaliação.


## Conclusão

Na avaliação independente (16 perguntas por modo), o agente apresentou a maior acurácia e a maior concordância nas recusas. O RAG simples obteve menor taxa de alucinação. O experimento é descritivo: base e amostra reduzidas, dados parcialmente simulados e um único conjunto de revisão humana. Um ID de fonte válido não garante fundamentação semântica da resposta.